# Standalone EDGAR 10-K Downloader for Colab

This notebook is self-contained. It does not import anything from the project repo.

Inputs:
- A ticker list text file in Google Drive, one ticker per line. Example: `company_ticker.txt`.

Outputs:
- Raw EDGAR files saved to a Google Drive folder you choose.
- Per-filing metadata JSON files.
- `_download_progress.json` and `_download_summary.csv` for cleaning/checking.
- Optional zip file for sharing with teammates.

Before running a large batch, set `USER_AGENT` to include your contact email. SEC automated access policies ask scripts to identify themselves.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip -q install requests tenacity pandas

In [ ]:
from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from tenacity import retry, stop_after_attempt, wait_exponential

## Paths and Settings

Change only this cell for your own Google Drive location.

In [ ]:
# Change these paths to your own Google Drive folders/files.
TICKER_FILE = Path('/content/drive/MyDrive/company_ticker.txt')
OUTPUT_ROOT = Path('/content/drive/MyDrive/edgar_10k_raw')

# SEC asks for a descriptive User-Agent with contact info.
USER_AGENT = 'UCI MSBA RAG Project your_email@example.com'

# How many most recent 10-K filings to download per company.
MAX_FILINGS_PER_TICKER = 10

# Be polite to SEC endpoints. 0.3 seconds follows the original project setting.
REQUEST_SLEEP_SECONDS = 0.3
TIMEOUT = 30

# If True, reruns skip filings whose metadata JSON already exists.
SKIP_EXISTING_METADATA = True

assert TICKER_FILE.exists(), f'Missing ticker file: {TICKER_FILE}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Ticker file: {TICKER_FILE}')
print(f'Output root: {OUTPUT_ROOT}')

## Downloader Code

Everything below is copied into the notebook so teammates can run this without the project repo.

In [ ]:
def make_headers(host: str | None = None) -> dict[str, str]:
    headers = {
        'User-Agent': USER_AGENT,
        'Accept-Encoding': 'gzip, deflate',
    }
    if host:
        headers['Host'] = host
    return headers


@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=1, min=1, max=20))
def http_get(url: str, headers: dict[str, str], timeout: int = TIMEOUT) -> requests.Response:
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()
    return resp


def safe_filename(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', name)


def cik_10digit(cik: str | int) -> str:
    return str(cik).zfill(10)


def cik_nopad(cik: str | int) -> str:
    return str(int(str(cik)))


def accession_no_dashes(accession: str) -> str:
    return accession.replace('-', '')


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, data: Any) -> None:
    ensure_dir(path.parent)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def write_text(path: Path, text: str) -> None:
    ensure_dir(path.parent)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(text)


def maybe_sleep() -> None:
    time.sleep(REQUEST_SLEEP_SECONDS)

In [ ]:
@dataclass
class FilingRecord:
    ticker: str
    cik: str
    company_name: str
    accession_number: str
    filing_date: str
    report_date: str | None
    form: str
    primary_document: str
    primary_doc_description: str | None

    @property
    def accession_nodash(self) -> str:
        return accession_no_dashes(self.accession_number)

    @property
    def filing_year(self) -> str:
        return self.filing_date[:4] if self.filing_date else 'unknown'

In [ ]:
def load_ticker_cik_mapping() -> dict[str, dict[str, str]]:
    print('Loading SEC ticker -> CIK mapping...')
    url = 'https://www.sec.gov/files/company_tickers.json'
    resp = http_get(url, headers=make_headers('www.sec.gov'))
    maybe_sleep()
    raw = resp.json()

    mapping: dict[str, dict[str, str]] = {}
    for _, item in raw.items():
        ticker = item['ticker'].upper()
        mapping[ticker] = {
            'cik_str': str(item['cik_str']),
            'title': item['title'],
        }
    return mapping


def load_company_submissions(cik: str) -> dict[str, Any]:
    cik_padded = cik_10digit(cik)
    url = f'https://data.sec.gov/submissions/CIK{cik_padded}.json'
    resp = http_get(url, headers=make_headers('data.sec.gov'))
    maybe_sleep()
    return resp.json()


def extract_recent_10k_filings(
    ticker: str,
    cik: str,
    company_name: str,
    submissions_json: dict[str, Any],
    max_filings: int = MAX_FILINGS_PER_TICKER,
) -> list[FilingRecord]:
    recent = submissions_json.get('filings', {}).get('recent', {})

    accession_numbers = recent.get('accessionNumber', [])
    filing_dates = recent.get('filingDate', [])
    report_dates = recent.get('reportDate', [])
    forms = recent.get('form', [])
    primary_documents = recent.get('primaryDocument', [])
    primary_doc_descs = recent.get('primaryDocDescription', [])

    filings: list[FilingRecord] = []
    for i, form in enumerate(forms):
        if form != '10-K':
            continue

        filing = FilingRecord(
            ticker=ticker,
            cik=cik,
            company_name=company_name,
            accession_number=accession_numbers[i],
            filing_date=filing_dates[i],
            report_date=report_dates[i] if i < len(report_dates) else None,
            form=form,
            primary_document=primary_documents[i],
            primary_doc_description=primary_doc_descs[i] if i < len(primary_doc_descs) else None,
        )
        filings.append(filing)

        if len(filings) >= max_filings:
            break

    return filings

In [ ]:
def build_archive_base_url(cik: str, accession_number: str) -> str:
    return (
        'https://www.sec.gov/Archives/edgar/data/'
        f'{cik_nopad(cik)}/{accession_no_dashes(accession_number)}'
    )


def download_primary_document(filing: FilingRecord, output_dir: Path) -> dict[str, str | None]:
    archive_base = build_archive_base_url(filing.cik, filing.accession_number)
    primary_doc_url = f'{archive_base}/{filing.primary_document}'

    ext = Path(filing.primary_document).suffix.lower() or '.html'
    filename_base = f'{filing.filing_year}_10K_{filing.filing_date}'
    primary_doc_path = output_dir / f'{filename_base}{ext}'

    print(f'    Downloading primary document: {primary_doc_url}')
    resp = http_get(primary_doc_url, headers=make_headers('www.sec.gov'))
    maybe_sleep()

    content_type = resp.headers.get('Content-Type', '')
    write_text(primary_doc_path, resp.text)

    return {
        'primary_doc_url': primary_doc_url,
        'primary_doc_path': str(primary_doc_path),
        'primary_doc_content_type': content_type,
    }


def try_download_full_submission_txt(filing: FilingRecord, output_dir: Path) -> dict[str, str | None]:
    archive_base = build_archive_base_url(filing.cik, filing.accession_number)
    txt_url = f'{archive_base}/{filing.accession_number}.txt'

    filename_base = f'{filing.filing_year}_10K_{filing.filing_date}_full_submission'
    txt_path = output_dir / f'{filename_base}.txt'

    try:
        print(f'    Trying full submission txt: {txt_url}')
        resp = http_get(txt_url, headers=make_headers('www.sec.gov'))
        maybe_sleep()
        write_text(txt_path, resp.text)
        return {
            'full_submission_txt_url': txt_url,
            'full_submission_txt_path': str(txt_path),
        }
    except Exception as exc:
        print(f'    [WARN] full submission txt not downloaded: {exc}')
        return {
            'full_submission_txt_url': None,
            'full_submission_txt_path': None,
        }


def save_filing_manifest(
    filing: FilingRecord,
    ticker_dir: Path,
    download_info: dict[str, str | None],
    txt_info: dict[str, str | None],
) -> None:
    manifest = {
        'ticker': filing.ticker,
        'company_name': filing.company_name,
        'cik': filing.cik,
        'form': filing.form,
        'filing_date': filing.filing_date,
        'report_date': filing.report_date,
        'filing_year': filing.filing_year,
        'accession_number': filing.accession_number,
        'accession_number_nodash': filing.accession_nodash,
        'primary_document': filing.primary_document,
        'primary_doc_description': filing.primary_doc_description,
        **download_info,
        **txt_info,
    }

    manifest_name = f'{filing.filing_year}_10K_{filing.filing_date}_metadata.json'
    write_json(ticker_dir / manifest_name, manifest)


def metadata_path_for_filing(filing: FilingRecord, ticker_dir: Path) -> Path:
    return ticker_dir / f'{filing.filing_year}_10K_{filing.filing_date}_metadata.json'

## Load Tickers

In [ ]:
def read_tickers(path: Path) -> list[str]:
    tickers = []
    seen = set()
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip().upper()
        if not line or line.startswith('#'):
            continue
        if line not in seen:
            tickers.append(line)
            seen.add(line)
    return tickers


tickers = read_tickers(TICKER_FILE)
print(f'Loaded {len(tickers)} unique tickers')
print(tickers[:30])

## Process One Ticker

In [ ]:
def process_ticker(ticker: str, mapping: dict[str, dict[str, str]]) -> dict[str, Any]:
    ticker = ticker.upper()
    status = {
        'ticker': ticker,
        'status': 'unknown',
        'filings_found': 0,
        'filings_downloaded': 0,
        'filings_skipped': 0,
        'filing_errors': 0,
        'error': None,
    }

    if ticker not in mapping:
        status['status'] = 'not_found_in_sec_mapping'
        print(f'[ERROR] {ticker}: not found in SEC ticker mapping')
        return status

    cik = mapping[ticker]['cik_str']
    company_name = mapping[ticker]['title']
    ticker_dir = OUTPUT_ROOT / ticker
    ensure_dir(ticker_dir)

    print(f'\n=== Processing {ticker} | CIK {cik} | {company_name} ===')
    try:
        submissions = load_company_submissions(cik)
        write_json(ticker_dir / 'submissions.json', submissions)

        filings = extract_recent_10k_filings(
            ticker=ticker,
            cik=cik,
            company_name=company_name,
            submissions_json=submissions,
            max_filings=MAX_FILINGS_PER_TICKER,
        )
        status['filings_found'] = len(filings)

        if not filings:
            status['status'] = 'no_10k_found'
            print(f'[WARN] {ticker}: no 10-K filings found in recent submissions')
            return status

        for filing in filings:
            manifest_path = metadata_path_for_filing(filing, ticker_dir)
            if SKIP_EXISTING_METADATA and manifest_path.exists():
                status['filings_skipped'] += 1
                print(f'  [SKIP] {filing.filing_date} | {filing.accession_number}')
                continue

            try:
                print(f'  [GET] {filing.filing_date} | {filing.accession_number}')
                download_info = download_primary_document(filing, ticker_dir)
                txt_info = try_download_full_submission_txt(filing, ticker_dir)
                save_filing_manifest(filing, ticker_dir, download_info, txt_info)
                status['filings_downloaded'] += 1
            except Exception as exc:
                status['filing_errors'] += 1
                print(f'  [ERROR] failed filing {filing.accession_number}: {exc}')

        status['status'] = 'ok' if status['filing_errors'] == 0 else 'partial_error'
        return status
    except Exception as exc:
        status['status'] = 'error'
        status['error'] = str(exc)
        print(f'[ERROR] {ticker}: {exc}')
        return status

## Smoke Test

Run this first to confirm paths and SEC access are working.

In [ ]:
SMOKE_TEST_N = 3

mapping = load_ticker_cik_mapping()
smoke_results = [process_ticker(ticker, mapping) for ticker in tickers[:SMOKE_TEST_N]]
pd.DataFrame(smoke_results)

## Full Download

This may take a long time for hundreds of companies. If Colab disconnects, rerun the setup cells and this cell. Existing metadata files are skipped when `SKIP_EXISTING_METADATA = True`.

In [ ]:
mapping = load_ticker_cik_mapping()
results = []

for idx, ticker in enumerate(tickers, start=1):
    print(f'\n##### {idx}/{len(tickers)} #####')
    result = process_ticker(ticker, mapping)
    results.append(result)

    progress_path = OUTPUT_ROOT / '_download_progress.json'
    write_json(progress_path, results)
    time.sleep(REQUEST_SLEEP_SECONDS)

print('Done.')

## Summary

In [ ]:
progress_path = OUTPUT_ROOT / '_download_progress.json'
if progress_path.exists():
    results = json.loads(progress_path.read_text(encoding='utf-8'))

summary = pd.DataFrame(results)
display(summary)
display(summary.groupby('status', dropna=False).size().rename('count').reset_index())

summary_path = OUTPUT_ROOT / '_download_summary.csv'
summary.to_csv(summary_path, index=False)
print(f'Saved summary to: {summary_path}')

## Zip Output for Sharing

In [ ]:
zip_path = OUTPUT_ROOT.with_suffix('.zip')
!cd "{OUTPUT_ROOT.parent}" && zip -qr "{zip_path.name}" "{OUTPUT_ROOT.name}"
print(f'Created: {zip_path}')